## Copying VoteNet

In [ ]:
import shutil
import os

SRC = "/kaggle/input/datasets/vathsal05/votenet-source/votenet_reference"
DST = "/kaggle/working/votenet"

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

print("VoteNet copied successfully.")

In [ ]:
%cd /kaggle/working/votenet

## Removing AppleDouble Files

In [ ]:
import os

removed = 0

for root, dirs, files in os.walk("/kaggle/working/votenet"):
    for f in files:
        if f.startswith("._"):
            os.remove(os.path.join(root, f))
            removed += 1

print(f"Removed {removed} AppleDouble files.")

## Verifying Repo Structure 

## Compiling PointNet2

In [ ]:
%cd /kaggle/working/votenet/pointnet2

In [ ]:
setup_py = r'''
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension
import glob
import os

this_dir = os.path.dirname(os.path.abspath(__file__))
ext_root = os.path.join(this_dir, "_ext_src")

sources = (
    glob.glob(os.path.join(ext_root, "src", "*.cpp")) +
    glob.glob(os.path.join(ext_root, "src", "*.cu"))
)

setup(
    name="pointnet2",
    ext_modules=[
        CUDAExtension(
            name="pointnet2._ext",
            sources=sources,
            include_dirs=[
                os.path.join(ext_root, "include"),
            ],
            extra_compile_args={
                "cxx": ["-O2"],
                "nvcc": ["-O2"],
            },
        )
    ],
    cmdclass={
        "build_ext": BuildExtension
    },
)
'''

with open("/kaggle/working/votenet/pointnet2/setup.py", "w") as f:
    f.write(setup_py)

print("✅ setup.py replaced")

In [ ]:
%cd /kaggle/working/votenet/pointnet2

!rm -rf build
!rm -rf pointnet2.egg-info
!find . -name "*.so" -delete

In [ ]:
%%bash
cd /kaggle/working/votenet

find pointnet2 -type f \( -name "*.cpp" -o -name "*.cu" -o -name "*.h" \) \
-exec sed -i 's/AT_CHECK/TORCH_CHECK/g' {} +

echo "Checking..."
grep -R "AT_CHECK" pointnet2 || echo "SUCCESS: No AT_CHECK remaining."

In [ ]:
%%bash
cd /kaggle/working/votenet/pointnet2
rm -rf build
rm -rf *.egg-info

In [ ]:
!python setup.py install

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/votenet/pointnet2/build/lib.linux-x86_64-cpython-312")

import torch
import pointnet2._ext as _ext

print("✅ Extension loaded successfully.")

## Verifying Datasets

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/vathsal05/votenet-v4-25d/synthetic_v4_25d")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "val"

for split, d in [("train", TRAIN_DIR), ("val", VAL_DIR)]:
    pcs = {p.stem.replace("_pc", "") for p in d.glob("*_pc.npz")
           if not p.name.startswith("._")}
    bbs = {p.stem.replace("_bbox", "") for p in d.glob("*_bbox.npy")
           if not p.name.startswith("._")}
    vts = {p.stem.replace("_votes", "") for p in d.glob("*_votes.npz")
           if not p.name.startswith("._")}
    print(f"{split}: pc={len(pcs)}  bbox={len(bbs)}  votes={len(vts)}  "
          f"complete={len(pcs & bbs & vts)}")
# expect: train 8000, val 1500 (real files only)


## Patch the VoteNet Backbone

In [ ]:
from pathlib import Path

BB = Path("/kaggle/working/votenet/models/backbone_module.py")

src = BB.read_text()

patches = [
(
"""npoint=2048,
                radius=0.2,
                nsample=64,""",
"""npoint=4096,
                radius=0.1,
                nsample=64,"""
),

(
"""npoint=1024,
                radius=0.4,
                nsample=32,""",
"""npoint=2048,
                radius=0.3,
                nsample=32,"""
),

(
"""npoint=512,
                radius=0.8,
                nsample=16,""",
"""npoint=1024,
                radius=0.6,
                nsample=16,"""
),

(
"""npoint=256,
                radius=1.2,
                nsample=16,""",
"""npoint=512,
                radius=1.2,
                nsample=16,"""
)
]

matched = 0

for old, new in patches:
    if old in src:
        src = src.replace(old, new)
        matched += 1

BB.write_text(src)

print(f"Matched {matched}/4 patterns")
assert matched == 4, f"only {matched}/4 backbone patterns matched - STOP, do not train"


In [ ]:
import shutil
import os

SRC = "/kaggle/input/datasets/vathsal05/votenet-v4-25d/synthetic_v4_25d"
DST = "/kaggle/working/synthetic_v4_25d"

if os.path.exists(DST):
    shutil.rmtree(DST)

print("Copying dataset...")
shutil.copytree(SRC, DST)

print("Done.")

In [ ]:
import os

removed = 0

for root, dirs, files in os.walk("/kaggle/working/synthetic_v4_25d"):
    for f in files:
        if f.startswith("._"):
            os.remove(os.path.join(root, f))
            removed += 1

print(f"Removed {removed} AppleDouble files.")

In [ ]:
import os

WORK = "/kaggle/working/phase10"

os.makedirs(WORK, exist_ok=True)

print(WORK)

In [ ]:
from pathlib import Path
import os

# ==========================================================
# Phase 10 Paths
# ==========================================================

WORK = Path("/kaggle/working/phase10")

# Clean dataset (AppleDouble files already removed)
DATA_ROOT = Path("/kaggle/working/synthetic_v4_25d")

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR   = DATA_ROOT / "val"

# Phase 9 checkpoint (60-class baseline)
CKPT_PATH = Path(
    "/kaggle/input/datasets/vathsal05/60-best/votenet_60class_best.pt"
)

# VoteNet repository
VOTENET_ROOT = Path("/kaggle/working/votenet")

print("="*60)
print("Phase 10 configuration")
print("="*60)

print("DATA_ROOT :", DATA_ROOT)
print("TRAIN_DIR :", TRAIN_DIR)
print("VAL_DIR   :", VAL_DIR)
print("CHECKPOINT:", CKPT_PATH)
print("VOTENET   :", VOTENET_ROOT)

assert DATA_ROOT.exists()
assert TRAIN_DIR.exists()
assert VAL_DIR.exists()

assert CKPT_PATH.exists()

assert VOTENET_ROOT.exists()

print("\nEverything verified.")

In [ ]:
import numpy as np
from pathlib import Path

CLASS_NAMES = [
    'bathtub','bed','bookshelf','chair','desk','dresser','night_stand','sofa','table','toilet',
    'ammo_box','binoculars','combat_knife','flashlight','gas_mask','hand_grenade','helmet','magazine','military_radio','pistol',
    'rifle','rocket_launcher','shotgun','sniper_rifle','tactical_backpack','tactical_vest','wire_cutter',
    'axe','barbed_wire_coil','baton','canteen','claymore_mine','concrete_barrier','crossbow','duffel_bag','entrenching_shovel',
    'field_telephone','first_aid_kit','flare_gun','fuel_drum','grenade_launcher','hedgehog','jerry_can','machete','machine_gun',
    'military_boots','military_cot','military_drone','military_shield','mortar_tube','night_vision_goggles','propane_tank','rifle_case',
    'sandbag','smoke_grenade','stretcher','submachine_gun','tank_mine','tank_shell','weapon_rack',
]

NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASS_NAMES)}

assert NUM_CLASSES == 60

print(f"{NUM_CLASSES} classes loaded.")

In [ ]:
raw = [
    l.rstrip()
    for l in open(DATA_ROOT / "classes.txt")
    if l.strip()
]

ref = [
    l.split("\t",1)[1].strip()
    if "\t" in l else l.strip()
    for l in raw
]

assert ref == CLASS_NAMES

print("Class order verified.")

In [ ]:
import numpy as np

sums = np.zeros((NUM_CLASSES,3))
counts = np.zeros(NUM_CLASSES,dtype=int)

scene_count = 0

for bbox_file in sorted(TRAIN_DIR.glob("*_bbox.npy")):

    if bbox_file.name.startswith("._"):
        continue

    scene_count += 1

    boxes = np.load(bbox_file)

    for box in boxes:
#
        cls = int(box[7])

        sums[cls] += np.array([
            box[3],
            box[5],
            box[4]
        ])

        counts[cls] += 1

mean_size_arr = sums / np.maximum(counts[:,None],1)

print(f"Processed {scene_count} training scenes.")

print()

print("Instance count")
print("--------------")
print("Minimum :", counts.min())
print("Maximum :", counts.max())
print("Median  :", int(np.median(counts)))

print()

print("Mean-size array ready.")

## Building SyntheticDatasetConfig

In [ ]:
# ==========================================================
# Phase 10 DatasetConfig
# ==========================================================

class SyntheticDatasetConfig:

    def __init__(self, mean_size_arr):

        self.num_class = NUM_CLASSES
        self.num_heading_bin = 12
        self.num_size_cluster = NUM_CLASSES

        self.class2type = {
            i: n for i, n in enumerate(CLASS_NAMES)
        }

        self.type2class = {
            n: i for i, n in enumerate(CLASS_NAMES)
        }

        self.type_mean_size = {
            n: mean_size_arr[i]
            for i, n in enumerate(CLASS_NAMES)
        }

        self.mean_size_arr = mean_size_arr.astype(np.float32)

    def size2class(self, size, type_name):
        size_class = self.type2class[type_name]
        size_residual = size - self.type_mean_size[type_name]
        return size_class, size_residual

    def class2size(self, pred_cls, residual):
        return self.mean_size_arr[pred_cls] + residual

    def angle2class(self, angle):

        angle = angle % (2*np.pi)

        angle_per_class = 2*np.pi / self.num_heading_bin

        shifted_angle = (angle + angle_per_class/2) % (2*np.pi)

        class_id = int(shifted_angle / angle_per_class)

        residual = shifted_angle - (
            class_id * angle_per_class + angle_per_class/2
        )

        return class_id, residual

    def class2angle(
        self,
        pred_cls,
        residual,
        to_label_format=True
    ):

        angle_per_class = 2*np.pi / self.num_heading_bin

        angle = pred_cls * angle_per_class + residual

        if to_label_format and angle > np.pi:
            angle -= 2*np.pi

        return angle

    def param2obb(
        self,
        center,
        heading_class,
        heading_residual,
        size_class,
        size_residual
    ):

        obb = np.zeros(7, dtype=np.float32)

        obb[:3] = center
        obb[3:6] = self.class2size(size_class, size_residual)
        obb[6] = self.class2angle(
            heading_class,
            heading_residual
        )

        return obb


DC = SyntheticDatasetConfig(mean_size_arr)

print("DatasetConfig ready")
print("Classes :", DC.num_class)
print("Size clusters :", DC.num_size_cluster)
print("Heading bins :", DC.num_heading_bin)

In [ ]:
peek = [
    "bed",
    "chair",
    "pistol",
    "rifle",
    "flashlight",
    "hand_grenade",
    "jerry_can",
    "sandbag",
    "concrete_barrier"
]

print("Mean object sizes (L × W × H)")

for c in peek:

    idx = CLASS_TO_IDX[c]

    s = mean_size_arr[idx]

    print(
        f"{c:22s}"
        f"{s[0]:6.2f}"
        f"{s[1]:6.2f}"
        f"{s[2]:6.2f}"
    )

In [ ]:
# Cell 5 — dataset (returns 4-channel pc) + dataloaders
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path

MAX_NUM_OBJ = 64

class SyntheticVoteNetDataset(Dataset):
    def __init__(self, split_dir, num_points=40000, augment=False, dataset_config=None):
        self.split_dir = Path(split_dir)
        self.num_points = num_points
        self.augment = augment
        self.DC = dataset_config
        self.scene_ids = sorted(
            f.name.replace('_pc.npz', '')
            for f in self.split_dir.glob('*_pc.npz')
            if not f.name.startswith('._')
        )
        print(f'{split_dir.name}: {len(self.scene_ids)} scenes')

    def __len__(self):
        return len(self.scene_ids)

    def __getitem__(self, idx):
        sid = self.scene_ids[idx]

        pc     = np.load(self.split_dir / f'{sid}_pc.npz')['pc'].astype(np.float32)
        bboxes = np.load(self.split_dir / f'{sid}_bbox.npy').astype(np.float32)
        votes  = np.load(self.split_dir / f'{sid}_votes.npz')['votes'].astype(np.float32)

        # ----------------------------------------------------
        # Random point sampling
        # ----------------------------------------------------
        N = pc.shape[0]
        if N != self.num_points:
            choice = np.random.choice(
                N,
                self.num_points,
                replace=(N < self.num_points)
            )
            pc = pc[choice]
            votes = votes[choice]

        # ----------------------------------------------------
        # Data augmentation
        # ----------------------------------------------------
        if self.augment:
            theta = np.random.uniform(-np.pi/6, np.pi/6)
            c, s = np.cos(theta), np.sin(theta)

            R = np.array([
                [ c, 0, s],
                [ 0, 1, 0],
                [-s, 0, c]
            ], dtype=np.float32)

            pc[:, :3] = pc[:, :3] @ R.T
            bboxes[:, :3] = bboxes[:, :3] @ R.T
            bboxes[:, 6] += theta

            for k in range(3):
                votes[:, 1+k*3:4+k*3] = votes[:, 1+k*3:4+k*3] @ R.T

            scale = np.random.uniform(0.9, 1.1)

            pc[:, :3] *= scale
            bboxes[:, :6] *= scale

            for k in range(3):
                votes[:, 1+k*3:4+k*3] *= scale

            if np.random.rand() < 0.5:
                pc[:, 0] = -pc[:, 0]
                bboxes[:, 0] = -bboxes[:, 0]
                bboxes[:, 6] = -bboxes[:, 6]

                for k in range(3):
                    votes[:, 1+k*3] = -votes[:, 1+k*3]

        # ----------------------------------------------------
        # Height feature (FIX)
        # Synthetic scenes are Y-up.
        # Compute height relative to the floor.
        # ----------------------------------------------------
        pc[:, 3] = pc[:, 1] - np.percentile(pc[:, 1], 0.99)

        # ----------------------------------------------------
        # Allocate labels
        # ----------------------------------------------------
        center_label = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        heading_class_label = np.zeros(MAX_NUM_OBJ, dtype=np.int64)
        heading_residual_label = np.zeros(MAX_NUM_OBJ, dtype=np.float32)
        size_class_label = np.zeros(MAX_NUM_OBJ, dtype=np.int64)
        size_residual_label = np.zeros((MAX_NUM_OBJ, 3), dtype=np.float32)
        sem_cls_label = np.zeros(MAX_NUM_OBJ, dtype=np.int64)
        box_label_mask = np.zeros(MAX_NUM_OBJ, dtype=np.float32)

        for i in range(min(len(bboxes), MAX_NUM_OBJ)):
            box = bboxes[i]
            cls = int(box[7])

            center_label[i] = box[:3]

            hc, hr = self.DC.angle2class(float(box[6]))
            heading_class_label[i] = hc
            heading_residual_label[i] = hr

            size_class_label[i] = cls

            reordered = np.array(
                [box[3], box[5], box[4]],
                dtype=np.float32
            )

            size_residual_label[i] = (
                reordered - self.DC.mean_size_arr[cls]
            )

            sem_cls_label[i] = cls
            box_label_mask[i] = 1

        return {
            'point_clouds': pc[:, :4].copy(),
            'center_label': center_label,
            'heading_class_label': heading_class_label,
            'heading_residual_label': heading_residual_label,
            'size_class_label': size_class_label,
            'size_residual_label': size_residual_label,
            'sem_cls_label': sem_cls_label,
            'box_label_mask': box_label_mask,
            'vote_label': votes[:, 1:10],
            'vote_label_mask': votes[:, 0].astype(np.int64),
            'scan_idx': np.array(idx, dtype=np.int64),
        }


train_ds = SyntheticVoteNetDataset(
    TRAIN_DIR,
    num_points=40000,
    augment=True,
    dataset_config=DC
)

val_ds = SyntheticVoteNetDataset(
    VAL_DIR,
    num_points=40000,
    augment=False,
    dataset_config=DC
)

sample = train_ds[0]

print(f'\nsample point_clouds shape: {sample["point_clouds"].shape} (should be (40000, 4))')
print(f'num valid boxes in sample 0: {int(sample["box_label_mask"].sum())}')

# Verify height feature
h = sample["point_clouds"][:, 3]
print(f'Height feature -> min: {h.min():.3f}, max: {h.max():.3f}, abs sum: {np.abs(h).sum():.1f}')
assert np.abs(h).sum() > 0, "Height feature is still all zeros!"
print("✅ Height feature successfully computed.")

In [ ]:
# Cell 6 — size-weighted semantic classification loss patch (IDEMPOTENT)

import numpy as np
import torch
import builtins
import importlib
import sys
import re

CLASS_REAL_SIZE = {
    'bed':2.0,'table':1.5,'sofa':2.0,'chair':0.55,'toilet':0.6,'desk':1.4,
    'dresser':1.0,'night_stand':0.55,'bookshelf':0.85,'bathtub':1.6,
    'ammo_box':0.35,'binoculars':0.22,'combat_knife':0.30,'flashlight':0.18,
    'gas_mask':0.28,'hand_grenade':0.12,'helmet':0.28,'magazine':0.18,
    'military_radio':0.30,'pistol':0.22,'rifle':0.95,'rocket_launcher':1.20,
    'shotgun':0.95,'sniper_rifle':1.20,'tactical_backpack':0.55,
    'tactical_vest':0.50,'wire_cutter':0.25,'axe':0.60,'barbed_wire_coil':0.90,
    'baton':0.55,'canteen':0.20,'claymore_mine':0.22,'concrete_barrier':2.00,
    'crossbow':0.75,'duffel_bag':0.80,'entrenching_shovel':0.60,
    'field_telephone':0.30,'first_aid_kit':0.30,'flare_gun':0.25,
    'fuel_drum':0.90,'grenade_launcher':0.75,'hedgehog':1.40,'jerry_can':0.47,
    'machete':0.65,'machine_gun':1.25,'military_boots':0.32,'military_cot':1.90,
    'military_drone':0.90,'military_shield':1.30,'mortar_tube':1.30,
    'night_vision_goggles':0.20,'propane_tank':0.60,'rifle_case':1.20,
    'sandbag':0.65,'smoke_grenade':0.15,'stretcher':2.10,'submachine_gun':0.60,
    'tank_mine':0.33,'tank_shell':0.90,'weapon_rack':1.80,
}

# -------------------------------------------------------
# Sanity check
# -------------------------------------------------------
missing = [c for c in CLASS_NAMES if c not in CLASS_REAL_SIZE]
assert not missing, f"Missing sizes for classes: {missing}"

sizes = np.array([CLASS_REAL_SIZE[c] for c in CLASS_NAMES])

# -------------------------------------------------------
# Compute semantic loss weights
# -------------------------------------------------------
w = np.clip(
    np.sqrt(np.median(sizes) / sizes),
    0.5,
    3.0
)

SEM_W = torch.tensor(
    w,
    dtype=torch.float32,
    device="cuda"
)

print(f"Median class size : {np.median(sizes):.2f} m")
print(f"Weight range      : [{w.min():.2f}, {w.max():.2f}]")

print("\nUP-weighted classes:")
for c, wt, sz in sorted(zip(CLASS_NAMES, w, sizes), key=lambda x: -x[1]):
    if wt > 1.5:
        print(f"  {c:24s} w={wt:.2f}   size={sz:.2f}m")

print("\nDOWN-weighted classes:")
for c, wt, sz in sorted(zip(CLASS_NAMES, w, sizes), key=lambda x: x[1]):
    if wt < 0.7:
        print(f"  {c:24s} w={wt:.2f}   size={sz:.2f}m")

# -------------------------------------------------------
# Patch loss_helper.py (safe to rerun)
# -------------------------------------------------------
LH = "/kaggle/working/votenet/models/loss_helper.py"

with open(LH, "r") as f:
    src = f.read()

old = "criterion_sem_cls = nn.CrossEntropyLoss(reduction='none')"

new = (
    "import builtins\n    "
    "criterion_sem_cls = nn.CrossEntropyLoss("
    "weight=getattr(builtins, 'SEM_CLS_WEIGHTS', None), "
    "reduction='none')"
)

if "SEM_CLS_WEIGHTS" in src:
    print("\nloss_helper.py already patched — skipping")

elif old in src:
    src = src.replace(old, new)

    with open(LH, "w") as f:
        f.write(src)

    print("\nloss_helper.py patched successfully")

else:
    print("\nCould not find original pattern.")
    print("Candidates found:\n")

    for m in re.finditer(
        r"criterion_sem_cls\s*=\s*nn\.CrossEntropyLoss[^)]*\)",
        src
    ):
        print(m.group(0))

    raise RuntimeError(
        "Pattern not found and file not already patched."
    )

# -------------------------------------------------------
# ALWAYS set weights
# -------------------------------------------------------
builtins.SEM_CLS_WEIGHTS = SEM_W

print(
    f"\nSEM_CLS_WEIGHTS installed "
    f"(shape={tuple(SEM_W.shape)}, device={SEM_W.device})"
)

# -------------------------------------------------------
# Reload modules if already imported
# -------------------------------------------------------
for module_name in ("loss_helper", "models.loss_helper"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"{module_name} reloaded")

print("\n✅ Weighted semantic classification loss is ACTIVE.")

In [ ]:
import sys
import subprocess

packages = [
    "plyfile",
    "tensorboardX",
    "trimesh",
    "opencv-python-headless",
    "easydict",
    "scikit-image"
]

print("Installing packages...\n")

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("\nInstallation complete.\n")

# Verify imports
mods = {
    "plyfile": "plyfile",
    "tensorboardX": "tensorboardX",
    "trimesh": "trimesh",
    "cv2": "opencv-python-headless",
    "easydict": "easydict",
    "skimage": "scikit-image",
}

print("Verification:")

for mod, pkg in mods.items():
    try:
        __import__(mod)
        print(f"✅ {mod}")
    except Exception as e:
        print(f"❌ {pkg}: {e}")

## Verification Gate (must pass before model build)

In [ ]:
# ==========================================================
# Cell 6.5 - VERIFICATION GATE (all must pass before Cell 7)
# ==========================================================
import os, builtins

BB = "/kaggle/working/votenet/models/backbone_module.py"
LH = "/kaggle/working/votenet/models/loss_helper.py"

bb = open(BB).read()
assert "npoint=4096" in bb and "radius=0.1" in bb, \
    "BACKBONE NOT PATCHED - re-run the patch cell"
print("backbone patch     : OK")

assert "SEM_CLS_WEIGHTS" in open(LH).read(), \
    "loss_helper NOT PATCHED - re-run Cell 6"
print("loss_helper patch  : OK")

assert getattr(builtins, "SEM_CLS_WEIGHTS", None) is not None, "re-run Cell 6"
print("SEM_CLS_WEIGHTS    : OK")

junk = sum(1 for r, d, fs in os.walk("/kaggle/working/votenet")
           for f in fs if f.startswith("._"))
assert junk == 0, f"{junk} AppleDouble files - re-run cleanup"
print("AppleDouble files  : OK (0)")

import pointnet2._ext
print("pointnet2 CUDA ext : OK")

s = train_ds[0]
assert abs(s["point_clouds"][:, 3]).sum() > 0, "height feature zero - Cell 5 fix missing"
print("height feature     : OK")

print("\nALL CHECKS PASSED - safe to build the model")


## Building VoteNet + Warm Start

In [ ]:
# ==========================================================
# Cell 7 - Build VoteNet + Warm Start (Phase 10, v2)
# ==========================================================
import os, sys, importlib
import torch

VOTENET = "/kaggle/working/votenet"
MODELS  = os.path.join(VOTENET, "models")
assert os.path.isfile(os.path.join(MODELS, "votenet.py")), \
    "votenet.py missing - re-run Cells 1-4 (copy + cleanup) first"

for p in [VOTENET, MODELS, os.path.join(VOTENET, "utils")]:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

from votenet import VoteNet

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

model = VoteNet(
    num_class=DC.num_class,
    num_heading_bin=DC.num_heading_bin,
    num_size_cluster=DC.num_size_cluster,
    mean_size_arr=DC.mean_size_arr,
    num_proposal=512,
    input_feature_dim=1,
    vote_factor=1,
    sampling='vote_fps'
).to(device)
print(f"VoteNet built : {sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters")

# ---- architecture assert: the patched backbone MUST be live ----
assert model.backbone_net.sa1.npoint == 4096, \
    "MODEL BUILT FROM UNPATCHED BACKBONE - the patch cell did not stick"
print("backbone geometry verified: SA1 npoint = 4096")

# ---- warm start from Phase 9 ----
CKPT_PATH = "/kaggle/input/datasets/vathsal05/60-best/votenet_60class_best.pt"
assert os.path.isfile(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
result = model.load_state_dict(state, strict=False)
print(f"Warm start loaded. Missing: {len(result.missing_keys)}  "
      f"Unexpected: {len(result.unexpected_keys)}")

# ---- forward-pass smoke test (height feature must be live) ----
model.eval()
sample = train_ds[0]
assert abs(sample["point_clouds"][:, 3]).sum() > 0, "height feature is zero"
inputs = {"point_clouds":
          torch.from_numpy(sample["point_clouds"]).unsqueeze(0).to(device)}
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    end_points = model(inputs)
print(f"Forward pass OK. Tensors: {len(end_points)}  "
      f"Peak GPU: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

os.makedirs("/kaggle/working/phase10", exist_ok=True)


In [ ]:
import os
os.chdir("/kaggle/working")
print(os.getcwd())

In [ ]:
# ==========================================================
# Cell 8 - Phase 10 training loop v2 - FINAL / COMMIT
# AdamW 1e-4, wd 1e-4 -> cosine 1e-6 | 40 epochs | clip 10.0
# session-safe resume | 7.8 h time budget | full logging
# NOTE: attach ONLY votenet-source, votenet-v4-25d, 60-best.
# Do NOT attach old notebook outputs (the resume glob would
# pick up the old stock-backbone checkpoint).
# ==========================================================
import builtins, glob as _glob, json, os, sys, time
import torch
from torch.utils.data import DataLoader

os.chdir("/kaggle/working")   # guard against stale cwd

for p in ["/kaggle/working/votenet",
          "/kaggle/working/votenet/models",
          "/kaggle/working/votenet/utils"]:
    if p not in sys.path:
        sys.path.insert(0, p)

from models.loss_helper import get_loss

# ---------------- config ----------------
EPOCHS        = 40
BATCH_SIZE    = 8
ACCUM_STEPS   = 1
BASE_LR       = 1e-4
MIN_LR        = 1e-6
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 10.0
NUM_WORKERS   = 2
TIME_BUDGET_H = 7.8
RUN_T0        = time.time()

WORK      = "/kaggle/working/phase10"
os.makedirs(WORK, exist_ok=True)
LAST_CKPT = f"{WORK}/votenet_60class_v4_last.pt"
BEST_CKPT = f"{WORK}/votenet_60class_v4_best.pt"
HIST_JSON = f"{WORK}/train_history.json"

device = torch.device("cuda:0")

assert getattr(builtins, "SEM_CLS_WEIGHTS", None) is not None, \
    "SEM_CLS_WEIGHTS missing - re-run Cell 6 before training."

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

b0 = next(iter(val_loader))
assert b0["point_clouds"][..., 3].abs().sum() > 0, "height feature is zero - pipeline bug"
assert model.backbone_net.sa1.npoint == 4096, "unpatched backbone - STOP"
print(f"Guards passed. {len(train_loader)} train / {len(val_loader)} val batches per epoch")

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=MIN_LR)

# ---------------- resume-if-exists ----------------
resume_path = next((c for c in
    [LAST_CKPT]
    + sorted(_glob.glob("/kaggle/input/**/votenet_60class_v4_last.pt",
                        recursive=True))
    if os.path.isfile(c)), None)

start_epoch, best_val, history = 0, float("inf"), []

if resume_path:
    print(f"WARNING: resuming from {resume_path} - make sure this is the")
    print("PATCHED-backbone run, not the old stock-backbone output!")
    ck = torch.load(resume_path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model_state_dict"])
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    scheduler.load_state_dict(ck["scheduler_state_dict"])
    start_epoch = ck["epoch"] + 1
    best_val    = ck["best_val_loss"]
    history     = ck.get("history", [])
    print(f"RESUMED at epoch {start_epoch} (best val {best_val:.4f})")
else:
    print("Fresh run - starting from the Phase 9 warm start loaded in Cell 7")

LOSS_KEYS = ["vote_loss", "objectness_loss", "box_loss", "sem_cls_loss"]

def run_epoch(loader, train):
    model.train() if train else model.eval()
    agg = {k: 0.0 for k in ["loss"] + LOSS_KEYS}
    obj_acc, nb = 0.0, 0
    if train:
        optimizer.zero_grad(set_to_none=True)
    for i, batch in enumerate(loader):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            end_points = model({"point_clouds": batch["point_clouds"]})
            for k, v in batch.items():
                if k not in end_points:
                    end_points[k] = v
            loss, end_points = get_loss(end_points, DC)
        if train:
            (loss / ACCUM_STEPS).backward()
            if (i + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        agg["loss"] += loss.item()
        for k in LOSS_KEYS:
            agg[k] += end_points[k].item()
        obj_acc += end_points["obj_acc"].item()
        nb += 1
    out = {k: v / nb for k, v in agg.items()}
    out["obj_acc"] = obj_acc / nb
    return out

for epoch in range(start_epoch, EPOCHS):
    elapsed_h = (time.time() - RUN_T0) / 3600
    epoch_h   = (history[-1]["min"] / 60) if history else 0.3
    if elapsed_h + epoch_h > TIME_BUDGET_H:
        print(f"Time budget hit at epoch {epoch} - stopping cleanly. "
              f"Add this version's output as input and re-run to resume.")
        break

    t0 = time.time()
    torch.cuda.reset_peak_memory_stats()

    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader,   train=False)
    scheduler.step()

    lr_now  = optimizer.param_groups[0]["lr"]
    dt_min  = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    history.append({"epoch": epoch, "lr": lr_now, "min": round(dt_min, 1),
                    "peak_gb": round(peak_gb, 2), "train": tr, "val": va})

    is_best = va["loss"] < best_val
    if is_best:
        best_val = va["loss"]

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val,
        "history": history,
    }, LAST_CKPT)

    if is_best:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "best_val_loss": best_val,
        }, BEST_CKPT)

    with open(HIST_JSON, "w") as f:
        json.dump(history, f, indent=1)

    print(f"[{epoch+1:02d}/{EPOCHS}] "
          f"train {tr['loss']:.4f} | val {va['loss']:.4f}{' *BEST*' if is_best else ''} | "
          f"vote {va['vote_loss']:.3f} obj {va['objectness_loss']:.3f} "
          f"box {va['box_loss']:.3f} sem {va['sem_cls_loss']:.3f} | "
          f"objacc {va['obj_acc']:.3f} | lr {lr_now:.2e} | "
          f"{dt_min:.1f} min | peak {peak_gb:.2f} GB")

    if epoch == start_epoch:
        remaining = EPOCHS - epoch - 1
        proj_h  = remaining * dt_min / 60
        total_h = (time.time() - RUN_T0) / 3600 + proj_h
        fits = ("FITS this session" if total_h <= TIME_BUDGET_H
                else "won't fit - will auto-stop; finish with one resume commit")
        print(f">>> projection: {dt_min:.1f} min/epoch x {remaining} left "
              f"~ {proj_h:.1f} h more (total ~ {total_h:.1f} h) -> {fits}")
        if peak_gb > 13.5:
            print(">>> WARNING: near T4 memory limit - BATCH_SIZE=4, ACCUM_STEPS=2")

print(f"\nDone. Best val loss: {best_val:.4f}")
print(f"BEST: {BEST_CKPT}")
print(f"LAST: {LAST_CKPT}")


## Evaluation (runs automatically after training, plan section 7)

In [ ]:
# ==========================================================
# Cell G0 - reload the BEST checkpoint from this run's training
# (the training loop leaves LAST-epoch weights in memory)
# ==========================================================
import os, torch

BEST = "/kaggle/working/phase10/votenet_60class_v4_best.pt"
assert os.path.isfile(BEST), "best checkpoint missing - did training run?"
assert model.backbone_net.sa1.npoint == 4096, "unpatched backbone - STOP"

ck = torch.load(BEST, map_location=device, weights_only=False)
model.load_state_dict(ck["model_state_dict"])   # strict
model.eval()
print(f"Evaluating BEST checkpoint: epoch {ck['epoch']}, "
      f"val loss {ck['best_val_loss']:.4f}")


In [ ]:
# ==========================================================
# Cell G1 - synthetic-val mAP@0.25 / @0.50 (publishable result #1)
# ==========================================================
import sys, torch
import numpy as np
from torch.utils.data import DataLoader

for p in ["/kaggle/working/votenet/models", "/kaggle/working/votenet/utils"]:
    if p not in sys.path:
        sys.path.insert(0, p)
from ap_helper import APCalculator, parse_predictions, parse_groundtruths

CONFIG_DICT = {
    "remove_empty_box": False, "use_3d_nms": True, "nms_iou": 0.25,
    "use_old_type_nms": False, "cls_nms": True, "per_class_proposal": True,
    "conf_thresh": 0.05, "dataset_config": DC,
}

eval_loader = DataLoader(val_ds, batch_size=8, shuffle=False,
                         num_workers=2, pin_memory=True)

ap25 = APCalculator(ap_iou_thresh=0.25, class2type_map=DC.class2type)
ap50 = APCalculator(ap_iou_thresh=0.50, class2type_map=DC.class2type)

all_preds, all_gts = [], []
model.eval()
for i, batch in enumerate(eval_loader):
    batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
    with torch.no_grad():
        end_points = model({"point_clouds": batch["point_clouds"]})
    for k, v in batch.items():
        if k not in end_points:
            end_points[k] = v
    batch_pred = parse_predictions(end_points, CONFIG_DICT)
    batch_gt   = parse_groundtruths(end_points, CONFIG_DICT)
    ap25.step(batch_pred, batch_gt)
    ap50.step(batch_pred, batch_gt)
    all_preds += batch_pred
    all_gts   += batch_gt
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(eval_loader)} batches")

m25, m50 = ap25.compute_metrics(), ap50.compute_metrics()
print(f"\n==============================")
print(f"mAP@0.25 = {m25['mAP']:.4f}   mAP@0.50 = {m50['mAP']:.4f}")
print(f"AR@0.25  = {m25['AR']:.4f}   AR@0.50  = {m50['AR']:.4f}")

def tier(c):
    s = CLASS_REAL_SIZE[c]
    return 1 if s >= 1.0 else (3 if s < 0.35 else 2)

rows = [(tier(c), c, m25.get(f"{c} Average Precision", 0.0))
        for c in CLASS_NAMES]
for t in (1, 2, 3):
    sub = [r for r in rows if r[0] == t]
    mean_ap = np.mean([r[2] for r in sub])
    print(f"\nTier {t} ({len(sub)} classes)  mean AP@0.25 = {mean_ap:.3f}")
    for _, c, a in sorted(sub, key=lambda r: -r[2]):
        print(f"  {c:24s} {a:.3f}")


In [ ]:
# ==========================================================
# Cell G2 - F1@10cm-center for the 19 small (<35 cm) classes
# ==========================================================
import numpy as np

SMALL = [c for c in CLASS_NAMES if CLASS_REAL_SIZE[c] < 0.35]
print(f"{len(SMALL)} small classes")
CONF_T, DIST_T = 0.5, 0.10

stats = {c: [0, 0, 0] for c in SMALL}   # TP, FP, FN

for preds, gts in zip(all_preds, all_gts):
    for cname in SMALL:
        ci = CLASS_TO_IDX[cname]
        p = [np.mean(cor, axis=0) for cls, cor, s in preds
             if cls == ci and s >= CONF_T]
        g = [np.mean(cor, axis=0) for cls, cor in gts if cls == ci]
        used, tp = set(), 0
        for gc in g:
            best, bd = -1, 1e9
            for j, pcen in enumerate(p):
                if j in used:
                    continue
                d = np.linalg.norm(np.asarray(pcen) - np.asarray(gc))
                if d < bd:
                    bd, best = d, j
            if best >= 0 and bd <= DIST_T:
                used.add(best)
                tp += 1
        stats[cname][0] += tp
        stats[cname][1] += len(p) - len(used)
        stats[cname][2] += len(g) - tp

print(f"\n{'class':24s} {'P':>6s} {'R':>6s} {'F1':>6s}   TP/FP/FN")
f1s = []
for c in SMALL:
    tp, fp, fn = stats[c]
    P  = tp / max(tp + fp, 1)
    R  = tp / max(tp + fn, 1)
    F1 = 2 * P * R / max(P + R, 1e-9)
    f1s.append(F1)
    print(f"{c:24s} {P:6.3f} {R:6.3f} {F1:6.3f}   {tp}/{fp}/{fn}")
print(f"\nmean small-object F1@10cm = {np.mean(f1s):.3f}")


In [ ]:
# ==========================================================
# Cell G3 - 60x60 confusion matrix + artifact dump
# ==========================================================
import numpy as np, pickle, os

CONF_T, MATCH_R = 0.5, 0.5
conf_mat = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
missed   = np.zeros(NUM_CLASSES, dtype=int)

for preds, gts in zip(all_preds, all_gts):
    P = [(cls, np.mean(cor, axis=0)) for cls, cor, s in preds if s >= CONF_T]
    for gcls, gcor in gts:
        gc = np.mean(gcor, axis=0)
        best, bd = None, 1e9
        for pcls, pcen in P:
            d = np.linalg.norm(pcen - gc)
            if d < bd:
                bd, best = d, pcls
        if best is not None and bd <= MATCH_R:
            conf_mat[gcls, best] += 1
        else:
            missed[gcls] += 1

OUT = "/kaggle/working/phase10_eval"
os.makedirs(OUT, exist_ok=True)
np.save(f"{OUT}/confusion_60.npy", conf_mat)
np.save(f"{OUT}/missed_60.npy", missed)

off = [(int(conf_mat[i, j]), CLASS_NAMES[i], CLASS_NAMES[j])
       for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
       if i != j and conf_mat[i, j] > 0]
print("Top 15 confusions (gt -> pred):")
for n, a, b in sorted(off, reverse=True)[:15]:
    print(f"  {a:22s} -> {b:22s} x{n}")

# dump predictions (score >= 0.25 to keep the pkl a sane size)
slim = [[(c, cor, s) for c, cor, s in pr if s >= 0.25] for pr in all_preds]
with open(f"{OUT}/val_predictions.pkl", "wb") as f:
    pickle.dump({"preds": slim, "gts": all_gts,
                 "map25": m25, "map50": m50}, f)
print("\nSaved to phase10_eval/: confusion_60.npy, missed_60.npy, val_predictions.pkl")
print("\n=== PHASE 10 TRAIN + EVAL COMPLETE ===")
